# Assignment Module 2: Pet Classification

The goal of this assignment is to implement a neural network that classifies images of 37 breeds of cats and dogs from the [Oxford-IIIT-Pet dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/). The assignment is divided into two parts: first, you will be asked to implement from scratch your own neural network for image classification; then, you will fine-tune a pretrained network provided by PyTorch.


## Dataset

The following cells contain the code to download and access the dataset you will be using in this assignment. Note that, although this dataset features each and every image from [Oxford-IIIT-Pet](https://www.robots.ox.ac.uk/~vgg/data/pets/), it uses a different train-val-test split than the original authors.


In [ ]:
!git clone https://github.com/CVLAB-Unibo/ipcv-assignment-2.git

In [ ]:
import os
import math
from pathlib import Path
from PIL import Image
from torch import Tensor
import torch
from torch.utils.data import Dataset
from typing import List, Tuple
import matplotlib.pyplot as plt
import wandb
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from tqdm.auto import tqdm


BATCH_SIZE = 4

In [ ]:
# Login into wandb
wandb.login()

In [27]:
LABELS_TO_NAME = {
    0: "Abyssinian",
    1: "american_bulldog",
    2: "american_pit_bull_terrier",
    3: "basset_hound",
    4: "beagle",
    5: "Bengal",
    6: "Birman",
    7: "Bombay",
    8: "boxer",
    9: "British_Shorthair",
    10: "chihuahua",
    11: "Egyptian_Mau",
    12: "english_cocker_spaniel",
    13: "english_setter",
    14: "german_shorthaired",
    15: "great_pyrenees",
    16: "havanese",
    17: "japanese_chin",
    18: "keeshond",
    19: "leonberger",
    20: "Maine_Coon",
    21: "miniature_pinscher",
    22: "newfoundland",
    23: "Persian",
    24: "pomeranian",
    25: "pug",
    26: "Ragdoll",
    27: "Russian_Blue",
    28: "saint_bernard",
    29: "samoyed",
    30: "scottish_terrier",
    31: "shiba_inu",
    32: "Siamese",
    33: "Sphynx",
    34: "staffordshire_bull_terrier",
    35: "wheaten_terrier",
    36: "yorkshire_terrier",
}

NUM_CLASSES = 37


class OxfordPetDataset(Dataset):
    def __init__(self, split: str, transform=None) -> None:
        super().__init__()

        self.root = Path("ipcv-assignment-2") / "dataset"
        self.split = split
        self.names, self.labels = self._get_names_and_labels()
        self.transform = transform

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx) -> Tuple[Tensor, int]:
        img_path = self.root / "images" / f"{self.names[idx]}.jpg"
        img = Image.open(img_path).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            img = self.transform(img)

        return img, label

    def get_num_classes(self) -> int:
        return max(self.labels) + 1

    def _get_names_and_labels(self) -> Tuple[List[str], List[int]]:
        names = []
        labels = []

        with open(self.root / "annotations" / f"{self.split}.txt") as f:
            for line in f:
                name, label = line.replace("\n", "").split(" ")
                names.append(name),
                labels.append(int(label) - 1)

        return names, labels

## Data Inspection


In [ ]:
preprocess = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_dataset = OxfordPetDataset("train", transform=preprocess)
val_dataset = OxfordPetDataset("val", transform=preprocess)
test_dataset = OxfordPetDataset("test", transform=preprocess)

train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_dl = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
def denorm_image(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    """
    Denormalize a tensor image (C, H, W) or (B, C, H, W) that was normalized
    using torchvision's Normalize(mean, std).
    """
    if tensor.ndim == 3:
        mean = torch.tensor(mean, device=tensor.device).view(-1, 1, 1)
        std = torch.tensor(std, device=tensor.device).view(-1, 1, 1)
    elif tensor.ndim == 4:
        mean = torch.tensor(mean, device=tensor.device).view(1, -1, 1, 1)
        std = torch.tensor(std, device=tensor.device).view(1, -1, 1, 1)
    else:
        raise ValueError("Expected tensor of shape (C,H,W) or (B,C,H,W)")

    return tensor * std + mean


print(f"The train dataset contains {len(train_dataset)}")
indexes = torch.randperm(len(train_dataset))[:9]
plt.figure(figsize=(15, 15))
for ind, i in enumerate(indexes):
    img, label = train_dataset[i]
    plt.subplot(3, 3, ind + 1)
    plt.imshow(denorm_image(img).permute(1, 2, 0))
    plt.title(f"Label= {LABELS_TO_NAME[label]}")
    plt.axis("off")

## Part 1: design your own network

Your goal is to implement a convolutional neural network for image classification and train it from scratch on `OxfordPetDataset`. You should consider yourselves satisfied once you obtain a classification accuracy on the test split of ~60%. You are free to achieve this however you want, except for a few rules you must follow:

- Compile this notebook by displaying the results obtained by the best model you found throughout your experimentation; then show how, by removing some of its components, its performance drops. In other words, do an _ablation study_ to prove that your design choices have a positive impact on the final result.

- Do not instantiate an off-the-self PyTorch network. Instead, construct your network as a composition of existing PyTorch layers. In more concrete terms, you can use e.g. `torch.nn.Linear`, but you cannot use e.g. `torchvision.models.alexnet`.

- Show your results and ablations with plots, tables, images, etc. — the clearer, the better.

Don't be too concerned with your model performance: the ~60% is just to give you an idea of when to stop. Keep in mind that a thoroughly justified model with lower accuracy will be rewarded more points than a poorly experimentally validated model with higher accuracy.


In [ ]:
class Block(nn.Module):
    def __init__(
        self,
        num_convs,
        input_channels,
        output_channels,
        use_relu: bool = True,
        use_batch_norm: bool = True,
        use_residual: bool = True,
    ):
        super().__init__()
        block = []
        for i in range(num_convs):
            block.append(
                nn.Conv2d(
                    input_channels if i == 0 else output_channels,
                    output_channels,
                    kernel_size=3,
                    padding="same",
                )
            )
            if use_batch_norm:
                block.append(nn.BatchNorm2d(output_channels))
            if use_relu and i != num_convs - 1:
                block.append(nn.ReLU())

        self.block = nn.Sequential(*block)
        self.use_relu = use_relu
        self.use_residual = use_residual
        if use_residual:
            self.conv1 = nn.Conv2d(input_channels, output_channels, kernel_size=1)

    def forward(self, x):
        # x: (B, input_channels, H, W)
        feats = self.block(x)
        # (B, output_channels, H, W)
        if self.use_residual:
            feats = feats + self.conv1(x)

        if self.use_relu:
            feats = nn.ReLU()(feats)

        return nn.MaxPool2d(3, stride=2)(feats)  # (B, output_channels, H/2, W/2)


class BaseNetwork(nn.Module):
    def __init__(
        self,
        num_classes,
        num_convs_per_block: int = 2,
        use_relu: bool = True,
        use_batch_norm: bool = True,
        use_residual: bool = True,
        num_blocks: int = 2,
        num_channels: int = 16,
        channels_factor: float = 2.0,
    ):
        super().__init__()

        blocks = []
        c_in = 3
        c_out = num_channels
        for i in range(num_blocks):
            blocks.append(
                Block(
                    num_convs_per_block,
                    c_in,
                    c_out,
                    use_relu,
                    use_batch_norm,
                    use_residual,
                )
            )
            c_in = c_out
            c_out = int(c_out * channels_factor)

        self.blocks = nn.Sequential(*blocks)

        self.fc = nn.Linear(c_in, num_classes)

    def forward(self, x):
        # x.shape = (B, c_in, 224, 224)
        feats = self.blocks(x)
        # feats.shape = (B, c_out, 224/(2^n), 224/(2^n)) where n is the num of blocks
        feats = nn.functional.adaptive_avg_pool2d(feats, 1)
        # feats.shape = (B, c_out, 1, 1)
        feats = torch.flatten(feats, 1)
        # feats.shape = (B, c_out)

        logits = self.fc(feats)
        # logits.shape(B, num_classes)
        return logits

In [ ]:
model = BaseNetwork(
    37,
    num_convs_per_block=2,
    use_relu=True,
    use_residual=True,
    use_batch_norm=True,
    num_blocks=4,
    num_channels=16,
    channels_factor=2,
)
# x=torch.randn(2,3,224,224)
feats, label = train_dataset[0]
feats = feats.unsqueeze(0)
print(feats.shape)
logits = model(feats)
print(logits.shape)
print(logits)

In [ ]:
def train(
    model: torch.nn.Module,
    dl_train,
    dl_val,
    criterion,
    optimizer,
    epochs: int,
    name: str,
    device: str = "cpu",
    project: str = "my_project",
    run_name: str = None,
    save_dir: str = "./checkpoints",
):
    # Initialize W&B run
    wandb.init(
        project=project,
        name=run_name,
        entity="fraktc-university-of-bologna",
        config={
            "epochs": epochs,
            "optimizer": optimizer.__class__.__name__,
            "criterion": criterion.__class__.__name__,
            "device": device,
            "model": model.__class__.__name__,
        },
    )

    # Ensure checkpoint directory exists
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(os.path.join(save_dir, name))

    model.to(device)
    global_step = 0
    best_val_acc = 0.0  # track best accuracy for saving best model

    for e in tqdm(range(epochs)):
        ### Training ###
        model.train()
        running_loss = 0.0

        for images, labels in dl_train:
            images, labels = images.to(device), labels.to(device)

            output_logits = model(images)
            loss = criterion(output_logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            global_step += 1

            # log train loss every 10 steps
            if global_step % 10 == 0:
                wandb.log({"train_loss_step": loss.item()}, step=global_step)
                running_loss = 0

        ### Validation ###
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in dl_val:
                images, labels = images.to(device), labels.to(device)

                output_logits = model(images)
                loss = criterion(output_logits, labels)
                val_loss += loss.item()

                pred_labels = output_logits.argmax(1)
                correct += (pred_labels == labels).sum().item()
                total += labels.size(0)

        avg_val_loss = val_loss / len(dl_val)
        val_accuracy = correct / total

        ### Logging ###
        wandb.log(
            {"val_loss_epoch": avg_val_loss, "val_accuracy": val_accuracy},
            step=len(dl_train) * (e + 1),
        )

        ### Checkpoint Saving ###
        # Save local checkpoint
        checkpoint_path = os.path.join(save_dir, name, f"epoch_{e+1}.pt")
        torch.save(
            {
                "epoch": e + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": avg_val_loss,
                "val_accuracy": val_accuracy,
            },
            checkpoint_path,
        )

        # Log model checkpoint to wandb
        wandb.save(checkpoint_path)

        # Save best model based on accuracy
        if val_accuracy > best_val_acc:
            best_val_acc = val_accuracy
            best_path = os.path.join(save_dir, name, "best_model.pt")
            torch.save(model.state_dict(), best_path)
            wandb.run.summary["best_val_accuracy"] = best_val_acc
            wandb.save(best_path)
            print(f"Saved new best model (epoch {e+1}, acc={best_val_acc:.4f})")

    wandb.finish()

In [ ]:
def test(
    model: torch.nn.Module,
    dl_test,
    criterion,
    checkpoint_path: str,
    device: str = "cpu",
    project: str = "my_project",
    run_name: str = "test_run",
):
    """
    Evaluate a trained model on the test set and compute performance metrics.

    Args:
        model: Trained torch.nn.Module
        dl_test: DataLoader for the test set
        criterion: Loss function
        checkpoint_path: Path to a saved model checkpoint (.pt)
        device: Device to run inference on ('cpu' or 'cuda')
        project: W&B project name
        run_name: W&B run name for logging results
    """

    # Initialize W&B
    wandb.init(
        project=project,
        name=run_name,
        entity="fraktc-university-of-bologna",
        job_type="test",
        config={
            "device": device,
            "model": model.__class__.__name__,
            "checkpoint": checkpoint_path,
        },
    )

    # Load model checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)  # if saved with model.state_dict() only

    model.to(device)
    model.eval()

    # Accumulators
    test_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dl_test:
            images, labels = images.to(device), labels.to(device)
            # (B, 3, 224, 224)
            # (B, N)
            # (B,)
            output_logits = model(images)
            loss = criterion(output_logits, labels)
            test_loss += loss.item()

            preds = output_logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Average loss
    avg_test_loss = test_loss / len(dl_test)

    # Metrics
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    rec = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
    f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)

    # Print report
    print("\n=== Test Results ===")
    print(f"Loss: {avg_test_loss:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1-score: {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds))

    # Log metrics to W&B
    wandb.log(
        {
            "test_loss": avg_test_loss,
            "test_accuracy": acc,
            "test_precision": prec,
            "test_recall": rec,
            "test_f1": f1,
            "confusion_matrix": wandb.plot.confusion_matrix(
                probs=None,
                y_true=all_labels,
                preds=all_preds,
                title="Confusion Matrix",
            ),
        }
    )

    wandb.finish()

    return {
        "test_loss": avg_test_loss,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "confusion_matrix": cm,
    }

## Definition of all the experiments to perform the ablation study


In [ ]:
BASE_PARAMETERS = {
    "num_classes": NUM_CLASSES,
    "num_convs_per_block": 4,
    "use_relu": True,
    "use_residual": True,
    "use_batch_norm": True,
    "num_blocks": 4,
    "num_channels": 32,
    "channels_factor": 2,
}

ALL_EXPERIMENTS = [
    {"name": "BaseNetwork", "params": {}},
    {"name": "NoReLU", "params": {"use_relu": False}},
    {"name": "NoResidual", "params": {"use_residual": False}},
    {"name": "NoBatchNorm", "params": {"use_batch_norm": False}},
    {
        "name": "FewerChannels",
        "params": {"channels_factor": math.sqrt(2), "num_channels": 16},
    },
    {"name": "FewerBlocks", "params": {"num_blocks": 2}},
    {"name": "FewerConvsPerBlocks", "params": {"num_convs_per_block": 2}},
]

In [ ]:
for experiment in ALL_EXPERIMENTS:
    p = BASE_PARAMETERS.copy()
    for to_overwrite in experiment["params"]:
        p[to_overwrite] = experiment["params"][to_overwrite]

    model = BaseNetwork(**p)

    # Print the number of parameters of the model
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model {experiment["name"]} has {num_params} trainable parameters")

In [ ]:
EPOCHS = 50
LR = 1e-3
WEIGHT_DECAY = 1e-4
OPTIMIZER = torch.optim.AdamW
CRITERION = nn.CrossEntropyLoss()

for experiment in ALL_EXPERIMENTS:
    p = BASE_PARAMETERS.copy()
    for to_overwrite in experiment["params"]:
        p[to_overwrite] = experiment["params"][to_overwrite]

    model = BaseNetwork(**p)

    # Train the model
    train(
        model=model,
        dl_train=train_dl,
        dl_val=val_dl,
        criterion=CRITERION,
        optimizer=OPTIMIZER(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY),
        epochs=EPOCHS,
        name=experiment["name"],
        device="cuda" if torch.cuda.is_available() else "cpu",
        project="IPCV2",
        run_name=f"{experiment["name"]}_run",
    )

In [ ]:
for experiment in ALL_EXPERIMENTS:
    p = BASE_PARAMETERS.copy()
    for to_overwrite in experiment["params"]:
        p[to_overwrite] = experiment["params"][to_overwrite]

    model = BaseNetwork(**p)

    test_metrics = test(
        model,
        test_dl,
        CRITERION,
        checkpoint_path=f"./checkpoint/{experiment["name"]}/best_model.pt",
    )

## Part 2: fine-tune an existing network

Your goal is to fine-tune a pretrained ResNet-18 model on `OxfordPetDataset`. Use the implementation provided by PyTorch, i.e. the opposite of part 1. Specifically, use the PyTorch ResNet-18 model pretrained on ImageNet-1K (V1). Divide your fine-tuning into two parts:

2A. First, fine-tune the ResNet-18 with the same training hyperparameters you used for your best model in part 1.

2B. Then, tweak the training hyperparameters in order to increase the accuracy on the test split. Justify your choices by analyzing the training plots and/or citing sources that guided you in your decisions — papers, blog posts, YouTube videos, or whatever else you may find useful. You should consider yourselves satisfied once you obtain a classification accuracy on the test split of ~90%.


In [ ]:
def fine_tune(
    model: torch.nn.Module,
    dl_train,
    dl_val,
    criterion,
    optimizer_full,
    optimizer_fc,
    epochs_full: int,
    epochs_fc: int,
    name: str,
    device: str = "cpu",
    project: str = "my_project",
    run_name: str = None,
    save_dir: str = "./checkpoints_finetune",
):
    # Train only the head
    model.requires_grad_ = False
    model.fc.requires_grad_ = True

    train(
        model,
        dl_train,
        dl_val,
        criterion,
        optimizer_fc,
        epochs_fc,
        name,
        device,
        project,
        run_name,
        save_dir,
    )

    # Train all together
    model.requires_grad_ = True
    train(
        model,
        dl_train,
        dl_val,
        criterion,
        optimizer_full,
        epochs_full,
        name,
        device,
        project,
        run_name,
        save_dir,
    )

In [28]:
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(in_features=512, out_features=NUM_CLASSES, bias=True)

c:\Users\franc\Desktop\Magistrale\IPCV\IPCV2\Lib\site-packages\torchvision\models\_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(


# 2.A


In [ ]:
EPOCHS_FC = 50
EPOCHS_FULL = 
LR = 1e-3
WEIGHT_DECAY = 1e-4
OPTIMIZER = torch.optim.AdamW
CRITERION = nn.CrossEntropyLoss()

fine_tune(
    model,
    train_dl,
    val_dl,
    criterion=CRITERION,
    optimizer_fc=OPTIMIZER(model.fc.parameters(), lr=LR, weight_decay=WEIGHT_DECAY),
    optimizer_full=OPTIMIZER(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY),
    epochs=EPOCHS,
    name="resnet18_base",
    device="cuda" if torch.cuda.is_available() else "cpu",
    project="IPCV2",
    run_name=f"{experiment["name"]}_run",
)

# 2.B
